In [63]:
import numpy as np
import scipy.integrate as integrate
import scipy.interpolate as interpolate
import matplotlib.pyplot as plt
from scipy.special import jn
from scipy.signal import find_peaks
from scipy.optimize import curve_fit
import cv2


In [64]:
def gaussian_2d(xy, amp, x0, y0, sigma_x, sigma_y):
    x, y = xy
    return amp * np.exp(-(((x - x0) ** 2) / (2 * sigma_x ** 2) + ((y - y0) ** 2) / (2 * sigma_y ** 2)))
  

def gaussian_fit(zdata,xdata,ydata):
    x, y = np.meshgrid(xdata, ydata)
    # Flatten the data for curve_fit
    xfit = x.ravel()
    yfit = y.ravel()
    zfit = zdata.ravel()
    # Estimate initial x0 and y0 based on the peak
    peak_index = np.unravel_index(np.argmax(zdata), zdata.shape)
    x0_guess = xdata[peak_index[1]]  # Corresponding x value
    y0_guess = ydata[peak_index[0]]  # Corresponding y value
    params_guess = (np.max(zdata),x0_guess,y0_guess,1,1)
    lower_bounds = [0, min(xdata), min(ydata), 0.1, 0.01]
    upper_bounds = [2000, max(xdata), max(ydata), 10, 10]
    # Fit the data
    popt, pcov = curve_fit(gaussian_2d, (xfit,yfit), zfit, p0=params_guess, bounds=(lower_bounds, upper_bounds), maxfev=10000)
    return popt

In [ ]:
def gauss(x,a,mu,s):
    return a*np.exp(-((x-mu)**2)/(2*s**2))

def custom_dilation(image):
    output = np.zeros_like(image)
    rows, cols = image.shape
    for r in range(1, rows - 1):
        for c in range(1, cols - 1):
            neighborhood = image[r-1:r+2, c-1:c+2]
            output[r, c] = np.max(neighborhood)
    return output

def super_locs(image):
    # Threshold finder
    histogram, bins = np.histogram(image.flatten(), bins=256, range=[0, 256])
    bins = np.delete(bins, -1)
    params, covariance = curve_fit(gauss, bins, histogram)
    threshold = params[1] + 4*params[2]

    # Max finder
    image_t = image.copy()
    image_t[image_t < threshold] = 0
    dilated_image = custom_dilation(image_t)
    image_f = image.copy()
    image_f[image_f != dilated_image] = 0
    posy, posx = np.nonzero(image_f)

    # Super loc
    number_of_part = len(posx)
    x_loc = np.zeros(number_of_part)
    y_loc = np.zeros(number_of_part)
    minus_step = 3
    plus_step = 4
    for i in range(number_of_part):
        #print(f"Max pixel position : {round(posx[i]),round(posy[i])}")
        x_idx = np.arange(round(posx[i]-minus_step),round(posx[i]+plus_step))
        y_idx = np.arange(round(posy[i]-minus_step),round(posy[i]+plus_step))
        center_part = image[y_idx, :][:, x_idx]
        params = gaussian_fit(center_part,x_idx, y_idx)
        #print(f"Super localized positions : {params[1], params[2]}")
        x_loc[i] = params[1]
        y_loc[i] = params[2]
    
    return x_loc, y_loc
    


image_path = 'Simulated_single_image_UINT8.jpg'
image = cv2.imread(image_path)
image = image[:,:,0]

x, y = super_locs(image)
print(x)
print(y)

[ 83.16565597 135.24504795  23.08306617  87.04316126  28.12335465]
[18.09438178 21.03770815 47.5850063  57.9361234  60.81028824]
